## 1. Setup

In [ ]:
!pip install -q timm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, time, math, shutil, random
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, timm
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from sklearn.metrics import confusion_matrix, classification_report

class CFG:
    drive_dir      = "/content/drive/MyDrive/OCT2017_MODIFIED"
    local_dir      = "/content/OCT2017/CellData/OCT"
    copy_from_drive = True
    out_dir        = "/content/drive/MyDrive/oct_mamba"
    img_size   = 128   # set to 128 for the small-image run
    run_name       = f"mamba_tl_{img_size}"   # keeps each size's checkpoint separate
    batch_size = 32
    backbone   = "convnext_tiny"   # pretrained on ImageNet
    d_model    = 256    # token width fed to Mamba
    depth      = 2      # number of Mamba blocks
    warmup_epochs   = 2      # stage 1: backbone frozen
    finetune_epochs = 6      # stage 2: backbone unfrozen
    lr_new      = 1e-3       # Mamba blocks and head, trained from scratch
    lr_backbone = 1e-4       # pretrained backbone, only used in stage 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

os.makedirs(CFG.out_dir, exist_ok=True)
CKPT_PATH = os.path.join(CFG.out_dir, CFG.run_name + "_best.pt")
print(torch.__version__, DEVICE)

2.11.0+cu128 cuda


In [ ]:
if CFG.copy_from_drive and not os.path.isdir(CFG.local_dir):
    print("copying from Drive...")
    shutil.copytree(CFG.drive_dir, CFG.local_dir)
else:
    print("using existing local copy")

TRAIN_DIR = os.path.join(CFG.local_dir, "train")
TEST_DIR = os.path.join(CFG.local_dir, "test")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CFG.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),   # left/right eye; never flip vertically
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(CFG.img_size * 256 / 224)),   # same 256:224 ratio at any size
    transforms.CenterCrop(CFG.img_size),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full = datasets.ImageFolder(TRAIN_DIR, train_tf)
n_val = int(len(full) * 0.1)
train_set, val_set = random_split(full, [len(full) - n_val, n_val],
                                  generator=torch.Generator().manual_seed(42))
# random_split hands back views onto the same dataset, so point the val view at an
# un-augmented copy. ImageFolder sorts files, so the indices still line up.
val_set.dataset = datasets.ImageFolder(TRAIN_DIR, eval_tf)
test_set = datasets.ImageFolder(TEST_DIR, eval_tf)

CLASSES = full.classes
def make_loader(dataset, shuffle):
    return DataLoader(dataset, CFG.batch_size, shuffle=shuffle, num_workers=4, pin_memory=True)

train_loader = make_loader(train_set, shuffle=True)
val_loader   = make_loader(val_set,   shuffle=False)
test_loader  = make_loader(test_set,  shuffle=False)

# how many images per class, just to sanity-check the data before training
labels = [label for _, label in full.samples]
counts_per_class = np.bincount(labels)
for name, count in zip(CLASSES, counts_per_class):
    print(name, count)

print(f"train {len(train_set)}  val {len(val_set)}  test {len(test_set)}")

copying from Drive...
CNV 1500
DME 1500
DRUSEN 1500
NORMAL 1500
train 5400  val 600  test 1000


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
class MambaBlock(nn.Module):
    def __init__(self, dim, d_state=16, d_conv=4, expand=2):
        super().__init__()
        inner = dim * expand
        self.n = d_state
        self.in_proj  = nn.Linear(dim, inner * 2, bias=False)
        self.conv     = nn.Conv1d(inner, inner, d_conv, padding=d_conv - 1, groups=inner)
        self.x_proj   = nn.Linear(inner, 1 + 2 * d_state, bias=False)   # delta, B, C
        self.dt_proj  = nn.Linear(1, inner)
        self.out_proj = nn.Linear(inner, dim, bias=False)
        A_init = torch.arange(1, d_state + 1).float().repeat(inner, 1)
        self.A_log = nn.Parameter(A_init.log())
        self.D = nn.Parameter(torch.ones(inner))

    def scan(self, u, delta, A, B, C):
        # u, delta: (batch, len, dim) | A: (dim, state) | B, C: (batch, len, state)
        dA = torch.exp(delta.unsqueeze(-1) * A)
        dB = delta.unsqueeze(-1) * B.unsqueeze(2) * u.unsqueeze(-1)
        h = u.new_zeros(u.shape[0], u.shape[2], self.n)
        out = []
        for t in range(u.shape[1]):
            h = dA[:, t] * h + dB[:, t]
            out.append((h * C[:, t].unsqueeze(1)).sum(-1))
        return torch.stack(out, 1) + u * self.D

    def forward(self, x):
        seq_len = x.shape[1]

        # split into the branch that goes through the scan and a parallel gate
        projected = self.in_proj(x)
        x, gate = projected.chunk(2, dim=-1)

        # depthwise conv mixes each token with its neighbours before the scan
        x = x.transpose(1, 2)
        x = self.conv(x)[:, :, :seq_len]
        x = x.transpose(1, 2)
        x = F.silu(x).float()

        # x_proj predicts the scan's per-token inputs: step size (delta) and B, C
        delta, B, C = self.x_proj(x).split([1, self.n, self.n], dim=-1)
        step = F.softplus(self.dt_proj(delta))
        A = -self.A_log.float().exp()

        y = self.scan(x, step, A, B, C)
        gated = y.to(gate.dtype) * F.silu(gate)
        return self.out_proj(gated)


class BiMamba(nn.Module):
    """A scan is causal, so each token only sees the ones before it. Running the
    sequence in both directions and averaging gives every token the full picture."""
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fwd, self.bwd = MambaBlock(dim), MambaBlock(dim)

    def forward(self, x):
        h = self.norm(x)
        return x + (self.fwd(h) + self.bwd(h.flip(1)).flip(1)) / 2

In [ ]:
class MambaTL(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(CFG.backbone, pretrained=pretrained,
                                          features_only=True, out_indices=(-1,))
        ch = self.backbone.feature_info.channels()[-1]
        self.proj   = nn.Linear(ch, CFG.d_model)
        self.blocks = nn.Sequential(*[BiMamba(CFG.d_model) for _ in range(CFG.depth)])
        self.norm   = nn.LayerNorm(CFG.d_model)
        self.head   = nn.Linear(CFG.d_model + ch, len(CLASSES))

    def forward(self, x):
        feature_map = self.backbone(x)[0]            # (batch, channels, h, w)  7x7 at 224, 4x4 at 128

        # turn the h x w grid into a sequence of tokens for Mamba (49 at 224, 16 at 128)
        tokens = feature_map.flatten(2)              # (batch, channels, h*w)
        tokens = tokens.transpose(1, 2)              # (batch, h*w, channels)
        tokens = self.proj(tokens)                   # (batch, h*w, d_model)

        tokens = self.blocks(tokens)
        tokens = self.norm(tokens)

        mamba_features = tokens.mean(dim=1)
        backbone_features = feature_map.mean(dim=(-2, -1))

        # Mamba branch + the backbone's own pooled features, so the model still works
        # if the Mamba branch underfits.
        combined = torch.cat([mamba_features, backbone_features], dim=-1)
        return self.head(combined)


model = MambaTL().to(DEVICE)
print("%.1fM parameters" % (sum(p.numel() for p in model.parameters()) / 1e6))

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

29.7M parameters


In [ ]:
!pip install -q torchinfo
from torchinfo import summary

summary(
    model,
    input_size=(1, 3, CFG.img_size, CFG.img_size),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=4
)

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
MambaTL                                            [1, 3, 128, 128]          [1, 4]                    --                        True
├─FeatureListNet: 1-1                              [1, 3, 128, 128]          [1, 768, 4, 4]            --                        True
│    └─Conv2d: 2-1                                 [1, 3, 128, 128]          [1, 96, 32, 32]           4,704                     True
│    └─LayerNorm2d: 2-2                            [1, 96, 32, 32]           [1, 96, 32, 32]           192                       True
│    └─ConvNeXtStage: 2-3                          [1, 96, 32, 32]           [1, 96, 32, 32]           --                        True
│    │    └─Identity: 3-1                          [1, 96, 32, 32]           [1, 96, 32, 32]           --                        --
│    │    └─Sequential: 3-2                        [1, 96, 

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)  # classes are balanced (1500/class)
scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))


@torch.no_grad()
def predict(net, dl):
    net.eval()
    preds, trues = [], []
    for x, y in dl:
        with torch.amp.autocast(DEVICE, enabled=scaler.is_enabled()):
            preds.append(net(x.to(DEVICE)).argmax(1).cpu())
        trues.append(y)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()


best = 0.0

def run_stage(name, epochs, train_backbone):
    global best

    # frozen weights get no gradients, so stage 1 is also cheaper in memory and time
    for param in model.backbone.parameters():
        param.requires_grad = train_backbone

    groups = [{"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")],
               "lr": CFG.lr_new}]
    if train_backbone:
        groups.append({"params": model.backbone.parameters(), "lr": CFG.lr_backbone})

    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * len(train_loader))

    for epoch in range(epochs):
        model.train()
        model.backbone.train(train_backbone)   # frozen backbone stays in eval mode
        start = time.time()
        for i, (x, y) in enumerate(train_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(DEVICE, enabled=scaler.is_enabled()):
                loss = criterion(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            if i % 200 == 0:
                print(f"  {i}/{len(train_loader)}  loss {loss.item():.4f}  {time.time()-start:.0f}s", flush=True)

        p, t = predict(model, val_loader)
        acc = (p == t).mean()
        print(f"[{name}] epoch {epoch+1}/{epochs}  val acc {acc:.4f}", flush=True)
        if acc > best:
            best = acc
            torch.save(model.state_dict(), CKPT_PATH)
            print("  saved")


run_stage("warm-up",   CFG.warmup_epochs,   train_backbone=False)
run_stage("fine-tune", CFG.finetune_epochs, train_backbone=True)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  0/169  loss 1.3416  24s
[warm-up] epoch 1/2  val acc 0.7850
  saved
  0/169  loss 0.6978  1s
[warm-up] epoch 2/2  val acc 0.8317
  saved
  0/169  loss 0.6650  9s


/tmp/ipykernel_528/2168110271.py:46: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


[fine-tune] epoch 1/6  val acc 0.2467
  0/169  loss 1.3606  1s
[fine-tune] epoch 2/6  val acc 0.2567
  0/169  loss 1.3820  2s
[fine-tune] epoch 3/6  val acc 0.9033
  saved
  0/169  loss 0.5328  1s
[fine-tune] epoch 4/6  val acc 0.9333
  saved
  0/169  loss 0.2243  1s
[fine-tune] epoch 5/6  val acc 0.9383
  saved
  0/169  loss 0.3898  1s
[fine-tune] epoch 6/6  val acc 0.9417
  saved


## 6. Test

Only 1000 images, so one image is worth 0.1 points. Read the per-class recall, not the
headline number — DRUSEN against CNV is where the remaining errors live.

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH))
p, t = predict(model, test_loader)

print("test accuracy %.4f\n" % (p == t).mean())
print(confusion_matrix(t, p), "\n")   # rows = true, columns = predicted
print(classification_report(t, p, target_names=CLASSES, digits=4))

test accuracy 0.8910

[[243   2   3   2]
 [  6 244   0   0]
 [ 69   2 173   6]
 [  0  12   7 231]] 

              precision    recall  f1-score   support

         CNV     0.7642    0.9720    0.8556       250
         DME     0.9385    0.9760    0.9569       250
      DRUSEN     0.9454    0.6920    0.7991       250
      NORMAL     0.9665    0.9240    0.9448       250

    accuracy                         0.8910      1000
   macro avg     0.9036    0.8910    0.8891      1000
weighted avg     0.9036    0.8910    0.8891      1000

